# 02 — Precisione, cancellazione e denaro
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

`decimal` ha coefficiente a 96 bit e scala 0–28: non è IEEE Decimal128. Nessun tipo finito rappresenta esattamente 1/3.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
Console.WriteLine($"double: {0.1+0.2:R}; decimal: {0.1m+0.2m}");
Console.WriteLine($"ULP(1): {double.BitIncrement(1)-1:R}; Epsilon: {double.Epsilon:R}");
double[] values = {1e16, 1, 1, -1e16};
Console.WriteLine(NumericAlgorithms.Sum<double>(values));
Console.WriteLine(NumericAlgorithms.KahanSum<double>(values));
Console.WriteLine(Precision.NearlyEqual(0.1+0.2,0.3));

In [ ]:
decimal[] lines = {0.005m,0.005m,0.005m};
foreach (var mode in new[] {MidpointRounding.ToEven,MidpointRounding.AwayFromZero})
    Console.WriteLine($"{mode}: righe={Precision.InvoiceTotal(lines,mode,true)}, totale={Precision.InvoiceTotal(lines,mode,false)}");
Console.WriteLine($"Scala 1.00m: {Precision.DecimalScale(1.00m)}");
Console.WriteLine(1m/3m);

## Esperimenti
Riordinare i dati: Kahan non è una somma esatta universale. Cambiare segno agli importi e scegliere una policy esplicita. Tolleranze assolute vicino a zero, relative per grandi valori. NaN non è uguale a sé stesso con ==. double.Epsilon è il minimo subnormale positivo, non la distanza tra 1 e il successivo double.